<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Step 5</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Step 5: The session</h1>
<p style="margin:.55em 0">A <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">Session</code> is one conversation, and <b>all</b> its state lives in our code: the agent's allowed tools, the
verified patient, the pending code, the shared-phone choice. Two big consequences:</p>
<ul style="margin:.4em 0;padding-left:1.4em">
  <li style="margin:.2em 0">guards don't depend on the model remembering anything;</li>
  <li style="margin:.2em 0">a dropped model connection can be replaced cheaply (step 7): the new one only needs the words so far.</li>
</ul>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Metrics first</h2><p style="margin:.55em 0">Every tool call is counted by skill and result; the server will serve these at <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">/metrics</code>.</p>

</div>

In [1]:
%%writefile build/app/metrics.py
"""In-process counters and gauges, served at /metrics in the Prometheus text format.

Enough for one process (dashboards, alerts on stalls/errors). Several processes behind a load balancer:
scrape each one, or swap this for prometheus_client; callers only use inc()/set().
"""
import threading
from collections import defaultdict

HELP = {
    "sessions_total": "Conversations started.",
    "sessions_active": "Conversations in progress.",
    "sessions_rejected_total": "Conversations refused because the server was full or shutting down.",
    "tool_calls_total": "Tool calls by skill and result status.",
    "stalls_total": "OpenAI connections that went silent mid-response.",
    "resumes_total": "Reconnects after a stall.",
    "session_errors_total": "Conversations that ended with an error, by kind.",
}


def _escape(v) -> str:
    return str(v).replace("\\", "\\\\").replace('"', '\\"').replace("\n", "\\n")


class Metrics:
    def __init__(self):
        self._lock = threading.Lock()  # skills (and so inc()) also run on worker threads
        self._values: dict[tuple, float] = defaultdict(float)

    def inc(self, name: str, by: float = 1, **labels) -> None:
        with self._lock:
            self._values[(name, tuple(sorted(labels.items())))] += by

    def set(self, name: str, value: float, **labels) -> None:
        with self._lock:
            self._values[(name, tuple(sorted(labels.items())))] = value

    def get(self, name: str, **labels) -> float:
        with self._lock:
            return self._values.get((name, tuple(sorted(labels.items()))), 0)

    def render(self) -> str:
        with self._lock:
            items = sorted(self._values.items())
        out, seen = [], set()
        for (name, labels), value in items:
            if name not in seen:
                seen.add(name)
                kind = "gauge" if name.endswith("_active") else "counter"
                out += [f"# HELP {name} {HELP.get(name, name)}", f"# TYPE {name} {kind}"]
            label = ",".join(f'{k}="{_escape(v)}"' for k, v in labels)
            out.append(f"{name}{{{label}}} {value:g}" if label else f"{name} {value:g}")
        return "\n".join(out) + "\n"


METRICS = Metrics()


Writing build/app/metrics.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The session</h2>
<p style="margin:.55em 0"><code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">instructions()</code> fills the agent's prompt file; unknown placeholders fail loudly. <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">run_tool</code> is the one
door from the model: the agent's allowlist, then the registry's guarded <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">call</code>, then metrics and a UI event.</p>

</div>

In [2]:
%%writefile build/app/session.py
"""A Session is one conversation. Everything the agent knows lives here, in our code, not in the model:
which tools this agent may use, which patient is verified, the one pending code, the shared-phone choice.
That is also what makes reconnects cheap: a new model connection only needs the words so far.
"""
import asyncio
import contextlib
import json
import logging
import uuid
from dataclasses import dataclass, field
from datetime import datetime
from typing import Callable

from app import registry
from app.domain import numbers
from app.metrics import METRICS
from app.settings import Settings, SettingsError
from app.verification import Otp, SimulatedSms

log = logging.getLogger(__name__)


class _Strict(dict):
    def __missing__(self, key):
        raise SettingsError(f"prompt uses unknown placeholder {{{key}}} (literal braces: write {{{{ and }}}})")


@dataclass
class Session:
    settings: Settings
    agent: str
    emit: Callable[[dict], None] = lambda m: None   # UI hook: every tool call, every SMS
    now: Callable[[], datetime] = datetime.now
    id: str = field(default_factory=lambda: uuid.uuid4().hex[:8])
    patient: dict | None = None                     # verified caller: {"id", "name"}
    choices: list | None = None                     # patients on a just-verified shared phone
    sms: object = None
    otp: Otp | None = None
    said: list = field(default_factory=list)        # the caller's latest words (transcripts), newest last

    def __post_init__(self):
        if self.agent not in self.settings.agents:
            raise SettingsError(f"unknown agent '{self.agent}'")
        self.config = self.settings.agents[self.agent]
        self.db = self.settings.path(self.settings.store.path)
        self.sms = self.sms or SimulatedSms(on_send=lambda m: self.emit({"type": "sms", **m}))
        o = self.settings.otp
        self.otp = self.otp or Otp(self.sms, self.settings.clinic.name, o.ttl_s, o.max_tries, o.max_sends)
        self.tools = registry.select(self.config.toolsets)  # this agent's allowlist
        self._unheard = 0                       # turns the caller finished whose transcript has not arrived yet
        self._heard = asyncio.Event()

    def hear(self, event: str, detail: str) -> None:
        """The engine's notes about the caller's speech: tool arguments are checked against their own words."""
        if event == "input_audio_buffer.speech_stopped":
            self._unheard += 1
        elif event in ("user_said", "user_unheard"):
            self._unheard = max(self._unheard - 1, 0)
            if detail:
                self.said = (self.said + [detail])[-3:]
            self._heard.set()

    async def caller_words(self) -> str:
        """The caller's latest words. Their transcript often lands just after the model's tool call: wait briefly."""
        if self._unheard:
            self._heard.clear()
            with contextlib.suppress(asyncio.TimeoutError):
                await asyncio.wait_for(self._heard.wait(), 1.5)
        return self.said[-1] if self.said else ""

    def state(self) -> dict:
        """What the UI (and a reconnect) needs to know. Never the code itself."""
        return {"patient": self.patient,
                "code_pending_for": self.otp.pending["purpose"] if self.otp.pending else None,
                "choosing_between": [c["name"] for c in self.choices] if self.choices else None}

    def specs(self) -> list[dict]:
        return [s.spec for s in self.tools.values()]

    def instructions(self) -> str:
        """The agent's prompt template (config) filled with today's date, the clinic and one line per skill."""
        template = self.settings.path(self.config.prompt).read_text(encoding="utf-8")
        today = self.now()
        lines = "\n".join(f"- {s.name}: {s.when}" for s in self.tools.values() if s.when)
        return template.format_map(_Strict(clinic=self.settings.clinic.name, skills=lines,
                                           today=f"{today:%A} {today.day} {today:%B %Y}")).strip()

    async def run_tool(self, name: str, arguments: str) -> dict:
        """The one door from the model to our code: allowlist, the caller's own digits, then registry.call
        (validate, guard, timeout)."""
        s = self.tools.get(name)
        try:
            args = json.loads(arguments or "{}")
        except ValueError:
            args = {}
        if isinstance(args, dict) and numbers.VALID.keys() & args.keys():
            if fixed := numbers.fix_digits(args, await self.caller_words()):
                log.info("%s: digits from the caller's words, not the model's hearing: %s", name, fixed)
                METRICS.inc("digits_corrected_total", skill=name)
                arguments = json.dumps(args)
        out = await registry.call(s, self, arguments, self.settings.skills.timeout_s) if s else \
            {"error": f"unknown tool {name!r}"}
        status = "error" if "error" in out else str(out.get("status", "ok"))
        METRICS.inc("tool_calls_total", skill=name if s else "unknown", status=status)
        self.emit({"type": "desk", "skill": name, "args": args if isinstance(args, dict) else {}, "result": out,
                   "state": self.state()})
        return out

    def context(self) -> str:
        """Our side of the conversation, for replay after a reconnect."""
        return f"Our state: {json.dumps(self.state())}"


Writing build/app/session.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Try it: a whole front-desk call, without a model</h2><p style="margin:.55em 0">Exactly the path a model's proposal takes.</p>

</div>

In [3]:
import sys
sys.path.insert(0, 'build')   # `import app` now means the app you are building

In [4]:
import json
from datetime import datetime
from app import registry, settings, store
from app.session import Session
from app.verification import Otp, SimulatedSms

s = settings.load(env={'APP__STORE__PATH': 'data/step5.db'})
store.init(s.path(s.store.path), seed_demo=True, reset=True)
registry.discover()
events, codes = [], iter(['111111', '222222'])
sms = SimulatedSms(on_send=lambda m: events.append({'type': 'sms', **m}))
ses = Session(s, 'front_desk', emit=events.append, sms=sms, otp=Otp(sms, 'City Care Clinic', gen=lambda: next(codes)))

async def call(tool, **args):
    return await ses.run_tool(tool, json.dumps(args))

print(ses.instructions()[:400], '...')

You are the voice front desk of City Care Clinic. Today is Tuesday 24 November 2026.
Be brief and warm. Ask one thing at a time.
Speak the caller's language: reply in the language they use (English, Hindi, Tamil, or a mix) and switch when they switch. Write Hindi in Devanagari script. Tool arguments stay in English: names in English letters, numbers as digits.

Rules:
- Use the tools for every fac ...


In [5]:
await call('my_appointments')                 # the guard answers, not the skill

{'status': 'needs_verification',
 'problem': 'the caller is not verified yet',
 'next': 'Ask for their mobile number and call verify_patient.'}

In [6]:
await call('get_order_status', order_id='A1')  # not this agent's tool

{'error': "unknown tool 'get_order_status'"}

In [7]:
await call('verify_patient', phone='98123 45678'), sms.outbox[-1]

({'status': 'code_sent',
  'sent_to': 'phone ending 5678',
  'next': 'Ask for the 6-digit code that was just texted.'},
 {'to': '+919812345678',
  'text': 'Your City Care Clinic verification code is 111111.'})

In [8]:
await call('confirm_code', code='111111')      # Asha and her son share the phone

{'status': 'choose_patient',
 'names': ['Asha Sharma', 'Rohan Sharma'],
 'next': 'Several patients share this phone: ask which of them they are calling about.'}

In [9]:
await call('choose_patient', name='Asha'), ses.state()

({'status': 'verified',
  'name': 'Asha Sharma',
  'next': 'Greet Asha, then carry on with what they already asked for (for example call my_appointments if they wanted their appointments). Ask only if they asked nothing yet.'},
 {'patient': {'id': 1, 'name': 'Asha Sharma'},
  'code_pending_for': None,
  'choosing_between': None})

In [10]:
await call('my_appointments')

{'patient': 'Asha Sharma',
 'appointments': [{'appointment_id': 1,
   'doctor': 'Dr. Anil Kapoor',
   'specialty': 'General Medicine',
   'date': 'Tuesday 1 December 2026',
   'time': '10:00 AM',
   'reason': 'fever and cough'}]}

In [11]:
[e['skill'] for e in events if e['type'] == 'desk'], any('111111' in json.dumps(e['result']) for e in events if e['type'] == 'desk')

(['my_appointments',
  'get_order_status',
  'verify_patient',
  'confirm_code',
  'choose_patient',
  'my_appointments'],
 False)

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0">The code reached the phone (the outbox) and never any tool result, which is all the model sees. <b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">06_realtime_client.ipynb</code>.</p>

</div>